In [ ]:
import csv
import os
from datetime import datetime

MOVIES_FILE = "movies.csv"
BOOKINGS_FILE = "bookings.csv"

# -------------------- DISPLAY FUNCTIONS --------------------

def line():
    print("─" * 72)


def header(title):
    print("\n")
    line()
    print(f"{title:^72}")
    line()


def pause():
    input("\nPress Enter to continue...")


# -------------------- FILE HANDLING --------------------

def initialize_files():
    if not os.path.exists(MOVIES_FILE):
        movies = [
            ["M001", "Avengers: Endgame", "Action", "14:00", "180"],
            ["M002", "Interstellar", "Sci-Fi", "17:00", "200"],
            ["M003", "Inception", "Thriller", "19:30", "220"],
            ["M004", "The Dark Knight", "Action", "21:00", "250"]
        ]

        with open(MOVIES_FILE, "w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["MovieID", "Movie", "Genre", "Time", "Price"])
            writer.writerows(movies)

    if not os.path.exists(BOOKINGS_FILE):
        with open(BOOKINGS_FILE, "w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([
                "BookingID",
                "Customer",
                "Phone",
                "MovieID",
                "Movie",
                "ShowTime",
                "Seats",
                "Quantity",
                "Amount",
                "BookingDate"
            ])


def load_movies():
    with open(MOVIES_FILE, "r", newline="") as file:
        return list(csv.DictReader(file))


def load_bookings():
    with open(BOOKINGS_FILE, "r", newline="") as file:
        return list(csv.DictReader(file))


# -------------------- MOVIE FUNCTIONS --------------------

def show_movies():
    header("NOW SHOWING")

    movies = load_movies()

    print(f"{'ID':<8}{'MOVIE':<25}{'GENRE':<15}{'TIME':<12}{'PRICE':>10}")
    line()

    for movie in movies:
        print(
            f"{movie['MovieID']:<8}"
            f"{movie['Movie']:<25}"
            f"{movie['Genre']:<15}"
            f"{movie['Time']:<12}"
            f"₹{movie['Price']:>8}"
        )

    line()


def select_movie():
    movies = load_movies()

    while True:
        movie_id = input("\nEnter Movie ID: ").strip().upper()

        for movie in movies:
            if movie["MovieID"] == movie_id:
                return movie

        print("❌ Invalid Movie ID. Please try again.")


# -------------------- SEAT FUNCTIONS --------------------

def get_booked_seats(movie_id, show_time):
    bookings = load_bookings()
    booked = []

    for booking in bookings:
        if (
            booking["MovieID"] == movie_id
            and booking["ShowTime"] == show_time
        ):
            if booking["Seats"]:
                booked.extend(booking["Seats"].split(","))

    return booked


def show_seats(movie_id, show_time):
    booked = get_booked_seats(movie_id, show_time)

    header("SEAT SELECTION")

    print("                 SCREEN")
    print("        ═══════════════════════")
    print()

    for row in "ABCDEF":
        for number in range(1, 9):
            seat = f"{row}{number}"

            if seat in booked:
                print("[XX]", end=" ")
            else:
                print(f"[{seat}]", end=" ")

        print()

    print("\n[XX] = Booked")
    print("[A1] = Available")
    print()


def select_seats(movie_id, show_time, quantity):
    booked = get_booked_seats(movie_id, show_time)

    show_seats(movie_id, show_time)

    while True:
        seats_input = input(
            f"Enter {quantity} seat(s), separated by comma: "
        ).upper().replace(" ", "")

        seats = seats_input.split(",")

        if len(seats) != quantity:
            print(f"❌ Please select exactly {quantity} seat(s).")
            continue

        valid = True

        for seat in seats:
            if len(seat) < 2:
                valid = False
                break

            row = seat[0]
            number = seat[1:]

            if row not in "ABCDEF" or not number.isdigit():
                valid = False
                break

            number = int(number)

            if number < 1 or number > 8:
                valid = False
                break

            if seat in booked:
                print(f"❌ Seat {seat} is already booked.")
                valid = False
                break

        if not valid:
            print("❌ Invalid seat selection.")
            continue

        if len(set(seats)) != len(seats):
            print("❌ Duplicate seats are not allowed.")
            continue

        return seats


# -------------------- BOOKING FUNCTION --------------------

def generate_booking_id():
    bookings = load_bookings()

    if not bookings:
        return "B001"

    numbers = []

    for booking in bookings:
        try:
            numbers.append(int(booking["BookingID"][1:]))
        except ValueError:
            pass

    if numbers:
        return f"B{max(numbers) + 1:03d}"

    return "B001"


def book_ticket():
    header("BOOK MOVIE TICKETS")

    show_movies()

    movie = select_movie()

    print(f"\nSelected Movie : {movie['Movie']}")
    print(f"Genre          : {movie['Genre']}")
    print(f"Show Time      : {movie['Time']}")
    print(f"Ticket Price   : ₹{movie['Price']}")

    customer = input("\nCustomer Name: ").strip()

    if not customer:
        print("❌ Customer name cannot be empty.")
        return

    phone = input("Phone Number: ").strip()

    if not phone.isdigit() or len(phone) != 10:
        print("❌ Please enter a valid 10-digit phone number.")
        return

    while True:
        try:
            quantity = int(input("Number of Tickets: "))

            if quantity < 1 or quantity > 10:
                print("❌ Select between 1 and 10 tickets.")
                continue

            break

        except ValueError:
            print("❌ Please enter a valid number.")

    seats = select_seats(
        movie["MovieID"],
        movie["Time"],
        quantity
    )

    price = float(movie["Price"])
    amount = price * quantity

    booking_id = generate_booking_id()
    booking_date = datetime.now().strftime("%d-%m-%Y %H:%M")

    print("\n")
    line()
    print("BOOKING SUMMARY".center(72))
    line()

    print(f"Booking ID    : {booking_id}")
    print(f"Customer      : {customer}")
    print(f"Movie         : {movie['Movie']}")
    print(f"Show Time     : {movie['Time']}")
    print(f"Seats         : {', '.join(seats)}")
    print(f"Tickets       : {quantity}")
    print(f"Price/Ticket  : ₹{price:.2f}")
    print(f"Total Amount  : ₹{amount:.2f}")

    line()

    confirm = input("Confirm booking? (Y/N): ").strip().upper()

    if confirm != "Y":
        print("❌ Booking cancelled.")
        return

    with open(BOOKINGS_FILE, "a", newline="") as file:
        writer = csv.writer(file)

        writer.writerow([
            booking_id,
            customer,
            phone,
            movie["MovieID"],
            movie["Movie"],
            movie["Time"],
            ",".join(seats),
            quantity,
            f"{amount:.2f}",
            booking_date
        ])

    print("\n" + "═" * 72)
    print("✓ BOOKING CONFIRMED".center(72))
    print("═" * 72)
    print(f"Booking ID : {booking_id}")
    print(f"Seats      : {', '.join(seats)}")
    print(f"Amount     : ₹{amount:.2f}")
    print("Enjoy your movie! 🎬")


# -------------------- VIEW BOOKINGS --------------------

def view_bookings():
    header("BOOKING HISTORY")

    bookings = load_bookings()

    if not bookings:
        print("No bookings found.")
        return

    print(
        f"{'ID':<8}"
        f"{'CUSTOMER':<18}"
        f"{'MOVIE':<23}"
        f"{'SEATS':<12}"
        f"{'AMOUNT':>10}"
    )

    line()

    for booking in bookings:
        movie = booking["Movie"]

        if len(movie) > 20:
            movie = movie[:20] + "..."

        print(
            f"{booking['BookingID']:<8}"
            f"{booking['Customer']:<18}"
            f"{movie:<23}"
            f"{booking['Seats']:<12}"
            f"₹{booking['Amount']:>8}"
        )

    line()


# -------------------- SEARCH BOOKING --------------------

def search_booking():
    header("SEARCH BOOKING")

    booking_id = input("Enter Booking ID: ").strip().upper()

    bookings = load_bookings()

    for booking in bookings:
        if booking["BookingID"] == booking_id:

            print()
            line()
            print("BOOKING DETAILS".center(72))
            line()

            print(f"Booking ID   : {booking['BookingID']}")
            print(f"Customer     : {booking['Customer']}")
            print(f"Phone        : {booking['Phone']}")
            print(f"Movie        : {booking['Movie']}")
            print(f"Show Time    : {booking['ShowTime']}")
            print(f"Seats        : {booking['Seats']}")
            print(f"Tickets      : {booking['Quantity']}")
            print(f"Amount       : ₹{booking['Amount']}")
            print(f"Booked On    : {booking['BookingDate']}")

            line()
            return

    print("❌ Booking not found.")


# -------------------- CANCEL BOOKING --------------------

def cancel_booking():
    header("CANCEL BOOKING")

    booking_id = input("Enter Booking ID: ").strip().upper()

    bookings = load_bookings()

    found = False
    new_bookings = []

    for booking in bookings:
        if booking["BookingID"] == booking_id:
            found = True

            print(f"\nBooking found for: {booking['Customer']}")
            print(f"Movie: {booking['Movie']}")
            print(f"Seats: {booking['Seats']}")

            confirm = input("\nConfirm cancellation? (Y/N): ")

            if confirm.upper() == "Y":
                continue
            else:
                new_bookings.append(booking)
        else:
            new_bookings.append(booking)

    if not found:
        print("❌ Booking not found.")
        return

    with open(BOOKINGS_FILE, "w", newline="") as file:
        fieldnames = [
            "BookingID",
            "Customer",
            "Phone",
            "MovieID",
            "Movie",
            "ShowTime",
            "Seats",
            "Quantity",
            "Amount",
            "BookingDate"
        ]

        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(new_bookings)

    print("✓ Booking cancelled successfully.")


# -------------------- MAIN MENU --------------------

def main():
    initialize_files()

    while True:

        print("\n")
        print("╔" + "═" * 70 + "╗")
        print("║" + "🎬 MOVIE TICKET BOOKING SYSTEM".center(70) + "║")
        print("╠" + "═" * 70 + "╣")
        print("║" + "1.  View Movies".ljust(70) + "║")
        print("║" + "2.  Book Movie Tickets".ljust(70) + "║")
        print("║" + "3.  View Booking History".ljust(70) + "║")
        print("║" + "4.  Search Booking".ljust(70) + "║")
        print("║" + "5.  Cancel Booking".ljust(70) + "║")
        print("║" + "6.  Exit".ljust(70) + "║")
        print("╚" + "═" * 70 + "╝")

        choice = input("\nEnter your choice (1-6): ").strip()

        if choice == "1":
            show_movies()
            pause()

        elif choice == "2":
            book_ticket()
            pause()

        elif choice == "3":
            view_bookings()
            pause()

        elif choice == "4":
            search_booking()
            pause()

        elif choice == "5":
            cancel_booking()
            pause()

        elif choice == "6":
            print("\n" + "═" * 72)
            print("Thank you for using Movie Ticket Booking System!".center(72))
            print("Goodbye! 🎬".center(72))
            print("═" * 72)
            break

        else:
            print("\n❌ Invalid choice. Please select 1-6.")


# -------------------- PROGRAM START --------------------

if __name__ == "__main__":
    main()



╔══════════════════════════════════════════════════════════════════════╗
║                    🎬 MOVIE TICKET BOOKING SYSTEM                     ║
╠══════════════════════════════════════════════════════════════════════╣
║1.  View Movies                                                       ║
║2.  Book Movie Tickets                                                ║
║3.  View Booking History                                              ║
║4.  Search Booking                                                    ║
║5.  Cancel Booking                                                    ║
║6.  Exit                                                              ║
╚══════════════════════════════════════════════════════════════════════╝

Enter your choice (1-6): 1


────────────────────────────────────────────────────────────────────────
                              NOW SHOWING                               
────────────────────────────────────────────────────────────────────────
ID      MOVIE      